# AllSortsHub Cartoon Studio — Episode 1 Colab Generator

T4 path uses CogVideoX-5B-I2V with INT8 quantization and CPU offload. This replaces the unreliable Wan GGUF loader on 15 GB T4 GPUs. Completed clips are checkpointed to Google Drive.

In [ ]:
# 1. GPU check
!nvidia-smi
import torch, shutil
if not torch.cuda.is_available(): raise RuntimeError('No CUDA GPU. Choose Runtime > Change runtime type > GPU.')
GPU_NAME=torch.cuda.get_device_name(0); VRAM_GB=torch.cuda.get_device_properties(0).total_memory/1024**3
print('PyTorch:',torch.__version__); print('GPU:',GPU_NAME); print('VRAM:',round(VRAM_GB,1),'GB'); print('FFmpeg:',shutil.which('ffmpeg'))
if VRAM_GB < 8: raise RuntimeError('At least 8 GB VRAM is required.')
USE_T4_PATH=VRAM_GB < 20
print('Generation path:', 'CogVideoX-5B-I2V INT8 T4-safe' if USE_T4_PATH else 'Wan 2.2 I2V-A14B')

In [ ]:
# 2. Persistent Drive + project
from google.colab import drive
drive.mount('/content/drive')
BASE='/content/drive/MyDrive/AllSortsHub-Wan2.2'
!mkdir -p "$BASE/models" "$BASE/generated" "$BASE/output"
%cd /content
!rm -rf cartoon-studio
!git clone -q https://github.com/parth01/AllSortsHub-Cartoon-Studio.git cartoon-studio
!python -m pip install -q -U 'diffusers>=0.35.0' transformers accelerate safetensors sentencepiece imageio-ffmpeg optimum-quanto
if USE_T4_PATH:
    !python -m pip install -q -U torchao
!apt-get update -qq && apt-get install -y -qq ffmpeg

In [ ]:
# 3. Configure T4-safe CogVideoX
import torch
if USE_T4_PATH:
    MODEL_ID='THUDM/CogVideoX-5b-I2V'
    COG_WIDTH,COG_HEIGHT,COG_FRAMES,COG_STEPS=720,480,49,30
    COG_DTYPE=torch.float16
    print(f'CogVideoX-5B-I2V INT8: {COG_WIDTH}x{COG_HEIGHT}, {COG_FRAMES} frames, {COG_STEPS} steps')
else:
    print('Using Wan 2.2 I2V-A14B for 24GB+ GPU')

In [ ]:
# 4. Load T4-safe I2V pipeline
import torch
from diffusers import CogVideoXImageToVideoPipeline, TorchAoConfig
from diffusers.quantizers import PipelineQuantizationConfig
from torchao.quantization import Int8WeightOnlyConfig
if USE_T4_PATH:
    quant_cfg=PipelineQuantizationConfig(quant_mapping={'transformer': TorchAoConfig(Int8WeightOnlyConfig())})
    print('Loading CogVideoX-5B-I2V with INT8 transformer quantization + CPU offload...')
    pipe=CogVideoXImageToVideoPipeline.from_pretrained(MODEL_ID, torch_dtype=COG_DTYPE, quantization_config=quant_cfg)
    pipe.enable_model_cpu_offload()
    pipe.vae.enable_tiling(); pipe.vae.enable_slicing()
    print('CogVideoX pipeline ready.')

In [ ]:
# 5. Prepare Episode 1 and restore checkpoints
from pathlib import Path
import json, shutil
ROOT=Path('/content/cartoon-studio/master-version/AllSortsHub-Billion 2'); LOCAL_GEN=ROOT/'wan_i2v'/'generated'; DRIVE_GEN=Path(BASE)/'generated'
LOCAL_GEN.mkdir(parents=True,exist_ok=True); DRIVE_GEN.mkdir(parents=True,exist_ok=True)
manifest=json.loads((ROOT/'wan_i2v'/'manifest.json').read_text())
for p in DRIVE_GEN.glob('shot_*.mp4'):
    t=LOCAL_GEN/p.name
    if not t.exists() or t.stat().st_size<10000: shutil.copy2(p,t)
print('Episode shots:',len(manifest['shots']))
print('Restored clips:',len(list(LOCAL_GEN.glob('shot_*.mp4'))))

In [ ]:
# 6. Resumable Episode 1 generation
import gc, shutil, torch
from diffusers.utils import export_to_video, load_image
prompts=(ROOT/'wan_i2v'/'prompts.txt').read_text()
STYLE='Modern 2D cel-shaded cartoon animation, bold clean black linework, semi-flat shading, vibrant colors, expressive facial acting, preserve the exact character designs and environment in the input image. Smooth readable hand-drawn motion. Keep faces, hair, clothing, proportions, props and background layout consistent.'
NEG='No photorealism, no 3D CGI, no live action, no extra fingers, no duplicate limbs, no warped faces, no character morphing, no costume changes, no hairstyle changes, no background replacement, no random objects, no random text, no logos, no watermark, no scene cuts, no extreme deformation.'
def prompt_for(n):
    marker=f'SHOT {n:02d} —'; s=prompts.find(marker); e=prompts.find('\n\nSHOT ',s+2); e=prompts.find('\n\nNEGATIVE',s+2) if e<0 else e
    if s<0: raise RuntimeError('Missing prompt for '+marker)
    return f'{STYLE} {prompts[s:e if e>=0 else None].split(chr(10),1)[1].strip()} {NEG}'
def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache(); torch.cuda.ipc_collect()
def run_cog(image,prompt,out,seed):
    if out.exists() and out.stat().st_size>10000: print('SKIP',out.name); return
    print('GENERATING',out.name,f'({COG_WIDTH}x{COG_HEIGHT}, {COG_FRAMES} frames, {COG_STEPS} steps, INT8)')
    cleanup_gpu()
    img=load_image(image=str(image)).convert('RGB').resize((COG_WIDTH,COG_HEIGHT))
    gen=torch.Generator(device='cuda').manual_seed(seed)
    try:
        video=pipe(prompt=prompt,image=img,num_videos_per_prompt=1,num_inference_steps=COG_STEPS,num_frames=COG_FRAMES,guidance_scale=6,generator=gen).frames[0]
        export_to_video(video,str(out),fps=8)
    except Exception as e:
        cleanup_gpu(); print('CogVideoX generation error:',repr(e)); raise
    cleanup_gpu()
for shot in manifest['shots']:
    n=int(shot['id']); image=ROOT/shot['image']; p=prompt_for(n); out=LOCAL_GEN/f'shot_{n:02d}.mp4'
    if not USE_T4_PATH: raise RuntimeError('This notebook revision is for the T4 CogVideoX path. For Wan 2.2 use a separate 24GB+ notebook.')
    run_cog(image,p,out,910000+n)
    if out.exists() and out.stat().st_size>10000: shutil.copy2(out,DRIVE_GEN/out.name); print('CHECKPOINT',out.name)
    cleanup_gpu()
print('Generation pass complete.')

In [ ]:
# 7. Assemble final Episode 1
%cd /content/cartoon-studio/master-version/AllSortsHub-Billion 2
!python3 wan_i2v/assemble_episode.py
!cp -f output/AllSortsHub_Episode_01_WAN_MASTER.mp4 "$BASE/output/"
!cp -f output/AllSortsHub_Episode_01_WAN_VERTICAL_9x16.mp4 "$BASE/output/"
!ls -lh output/AllSortsHub_Episode_01_WAN_MASTER.mp4 output/AllSortsHub_Episode_01_WAN_VERTICAL_9x16.mp4

## If Colab disconnects
Reconnect to a GPU, rerun cells 1–5, then rerun cell 6. Completed MP4s in `MyDrive/AllSortsHub-Wan2.2/generated/` are restored and skipped.

The T4 path uses CogVideoX-5B-I2V with INT8 transformer quantization, CPU offload, VAE tiling/slicing, 720×480 and 49 frames. Quantized CogVideoX is documented as usable on free T4 Colab GPUs.